# Running multiple circuits with `CircuitBinding`

`CircuitBinding` groups circuits, parameter values, and measurements into related
runs. Providers can package these runs into one submission and, when compatible,
share hardware executions to reduce QPU time and cost. Providers without native
support execute the binding as a standard batch.

The three **axes** of a binding are its circuits, parameter value sets, and
measurements. Each logical execution selects one entry from each axis.

A binding lets you:

- Run several circuits together.
- Evaluate a circuit at different parameter values.
- Measure the same circuit with different observables.
- Combine these patterns using **PRODUCT**, **ZIP**, or nested bindings.

This is MPQP's counterpart to Qiskit's PUBs and Braket's ProgramSets. The examples
below use local simulators.

In [1]:
# First we setup here the circuits, values and observables to be used
from mpqp import BindingMode, CircuitBinding
from mpqp.execution.devices import (
    IBMDevice,
    AWSDevice,
)

from mpqp import (
    CNOT,
    BasisMeasure,
    ExpectationMeasure,
    H,
    IBMDevice,
    Observable,
    QCircuit,
    U,
    pI,
    pZ,
    pX,
    run,
)
from sympy import Symbol
from mpqp.execution.runner import run
from mpqp.execution.devices import IBMDevice, AWSDevice
import numpy as np

theta, phi, psi = Symbol('θ'), Symbol('phi'), Symbol('psi')
c1 = QCircuit([U(theta, phi, psi, 0)], label="c1")
c2 = QCircuit([H(0)], label="c2")
c3 = QCircuit([H(0), CNOT(0, 1)], label="c3")
c4 = QCircuit([H(0), H(1), CNOT(0, 1)], label="c4")

v1 = {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
v2 = {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
v3 = {'θ': 3.0, 'phi': 3.0, 'psi': 3.0}
v4 = {'θ': 4.0, 'phi': 4.0, 'psi': 4.0}
v_t0 = {'θ': 0}
v_t1 = {'θ': np.pi}

m1 = ExpectationMeasure(Observable(pI), label="Exp1", shots=2024)
m1_bis = ExpectationMeasure(Observable(pZ), label="Exp1bis", shots=2024)
m2 = ExpectationMeasure(Observable(pX @ pZ), label="Exp2", shots=2024)
m2_bis = ExpectationMeasure(Observable(pZ @ pZ), label="Exp2bis", shots=2024)
m3 = BasisMeasure(label="b3", shots=2024)
m3_bis = BasisMeasure(label="b3_bis", shots=1024)
m4 = None

m_I = ExpectationMeasure(Observable(pI), label="Exp_I", shots=2024)
m_Z = ExpectationMeasure(Observable(pZ), label="Exp_Z", shots=2024)

## 1. Run a list of circuits

For a basic batch, pass the circuits directly to `CircuitBinding`.

- **State vectors:** leave the circuits without measurements.
- **Samples:** include a `BasisMeasure` in each circuit.
- **Expectation values:** include an `ExpectationMeasure` in each circuit.

You can also supply measurements through the binding's `measurements` argument,
provided the circuits do not already contain measurements. The sampling example
below shows both forms.

In [2]:
# Example 1: Two state vector jobs
# Note: state vector jobs are not supported by braket's ProgramSets
binding = CircuitBinding([c3, c4])
print(run(binding, IBMDevice.AER_SIMULATOR))

BatchResult: 2 results
    Result: c3, IBMDevice, AER_SIMULATOR
      State vector: [0.70711, 0, 0, 0.70711]
      Probabilities: [0.5, 0, 0, 0.5]
      Number of qubits: 2
    Result: c4, IBMDevice, AER_SIMULATOR
      State vector: [0.5, 0.5, 0.5, 0.5]
      Probabilities: [0.25, 0.25, 0.25, 0.25]
      Number of qubits: 2


In [3]:
binding_with_basismeasure = CircuitBinding([c3 + QCircuit([m3]), c4 + QCircuit([m3])])
# or you can do
binding_with_basismeasure2 = CircuitBinding(
    [c3, c4], measurements=[m3], mode=BindingMode.PRODUCT
)
print(run(binding_with_basismeasure, AWSDevice.BRAKET_LOCAL_SIMULATOR))

BatchResult: 2 results
    Result: c3, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Counts: [1010, 0, 0, 1014]
      Probabilities: [0.49901, 0, 0, 0.50099]
      Samples:
        State: 00, Index: 0, Count: 1010, Probability: 0.4990119
        State: 11, Index: 3, Count: 1014, Probability: 0.5009881
      Error: None
    Result: c4, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Counts: [515, 527, 462, 520]
      Probabilities: [0.25445, 0.26038, 0.22826, 0.25692]
      Samples:
        State: 00, Index: 0, Count: 515, Probability: 0.2544466
        State: 01, Index: 1, Count: 527, Probability: 0.2603755
        State: 10, Index: 2, Count: 462, Probability: 0.2282609
        State: 11, Index: 3, Count: 520, Probability: 0.256917
      Error: None


In [4]:
binding_with_observables = CircuitBinding([c3 + QCircuit([m2]), c4 + QCircuit([m2])])

print(run(binding_with_observables, IBMDevice.AER_SIMULATOR))
print(run(binding_with_observables, AWSDevice.BRAKET_LOCAL_SIMULATOR))

BatchResult: 2 results
    Result: c3, IBMDevice, AER_SIMULATOR
      Expectation value: 0.023209876543209877
      Error/Variance: 0.0222162358784702
      Observables: [pX@pZ]
    Result: c4, IBMDevice, AER_SIMULATOR
      Expectation value: -0.009382716049382716
      Error/Variance: 0.02222124403002106
      Observables: [pX@pZ]
BatchResult: 2 results
    Result: c3, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Expectation value: 0.003952569169960474
      Error/Variance: 0.0004940634274688521
      Observables: [pX@pZ]
    Result: c4, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Expectation value: 0.016798418972332016
      Error/Variance: 0.0004939317258498172
      Observables: [pX@pZ]


## 2. Choose a binding mode

### PRODUCT: evaluate every combination

`BindingMode.PRODUCT` combines every circuit with every parameter set and every
measurement. It is the default mode.

For example:

```python
CircuitBinding(
    [c1, c2],
    values=[v1, v2],
    measurements=[m1, m1_bis],
    mode=BindingMode.PRODUCT,
)
```

This produces **8 logical executions**, in the following order:

| Circuit | Parameter values | Measurement |
| --- | --- | --- |
| `c1` | `v1` | `m1` |
| `c1` | `v1` | `m1_bis` |
| `c1` | `v2` | `m1` |
| `c1` | `v2` | `m1_bis` |
| `c2` | `v1` | `m1` |
| `c2` | `v1` | `m1_bis` |
| `c2` | `v2` | `m1` |
| `c2` | `v2` | `m1_bis` |

The runnable example below uses only `v1`, giving **4 logical executions**.
The number of logical executions need not equal the number of hardware executions.

In [5]:
binding_product = CircuitBinding(
    [c1, c2], values=[v1], measurements=[m1, m1_bis], mode=BindingMode.PRODUCT
)

print(run(binding_product, AWSDevice.BRAKET_LOCAL_SIMULATOR))

BatchResult: 4 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 0.532608695652174
      Error/Variance: 0.0003539169848397677
      Observables: [pZ]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: -0.0266798418972332
      Error/Variance: 0.0004937194595041199
      Observables: [pZ]


### ZIP: pair corresponding entries

`BindingMode.ZIP` pairs entries from the three axes by position. Each axis must
have either **one entry** or **as many entries as the longest axis**. An axis with
one entry is broadcast: its entry is reused for every position.

```python
CircuitBinding(
    [c1, c2],
    values=[v1, v2],
    measurements=[m1, m1_bis],
    mode=BindingMode.ZIP,
)
```

This produces **2 logical executions**:

| Circuit | Parameter values | Measurement |
| --- | --- | --- |
| `c1` | `v1` | `m1` |
| `c2` | `v2` | `m1_bis` |

For example, a circuit axis of length one is reused for every pair of values and
measurements:

```python
CircuitBinding(
    [c1],
    values=[v1, v2],
    measurements=[m1, m1_bis],
    mode=BindingMode.ZIP,
)
```

| Circuit | Parameter values | Measurement |
| --- | --- | --- |
| `c1` | `v1` | `m1` |
| `c1` | `v2` | `m1_bis` |

The same broadcasting rule applies to **all three axes**.

An example of failing inputs would be inputs of lengths `(2, 3, 1)`. They are 
incompatible: the circuit axis has neither one entry nor the maximum length, 
three. Invalid lengths raise `ValueError` when the binding is constructed. 

Lastly, as you will see bellow, an omitted values or measurements axis counts as
one unspecified entry.

The next two cells first pair individual circuits, then reuse a nested binding
containing both circuits for each pair.

In [6]:
binding_zip = CircuitBinding(
    [c1, c2], values=[v1, v2], measurements=[m1, m1_bis], mode=BindingMode.ZIP
)

print(run(binding_zip, AWSDevice.BRAKET_LOCAL_SIMULATOR))

BatchResult: 2 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.023715415019762844
      Error/Variance: 0.000493793270301601
      Observables: [pZ]


In [7]:
binding_zip_1_circuit = CircuitBinding(
    [CircuitBinding([c1, c2])],
    values=[v1, v2],
    measurements=[m1, m1_bis],
    mode=BindingMode.ZIP,
)

print(run(binding_zip_1_circuit, AWSDevice.BRAKET_LOCAL_SIMULATOR))

BatchResult: 4 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.39822134387351776
      Error/Variance: 0.00041572122593061734
      Observables: [pZ]
    Result: c2, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.001976284584980237
      Error/Variance: 0.0004940692165510075
      Observables: [pZ]


**A nested binding counts as one entry in the outer circuit list.** In the
preceding example, `[CircuitBinding([c1, c2])]` is therefore reused for both
value/measurement pairs, applying each pair to both inner circuits.

The next cell shows the simpler case: reuse only `c1` for the two pairs.

In [8]:
binding_zip_1_cb = CircuitBinding([c1], [v1, v2], [m1, m1_bis], mode=BindingMode.ZIP)

print(run(binding_zip_1_cb, AWSDevice.BRAKET_LOCAL_SIMULATOR))

BatchResult: 2 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}
      Expectation value: 1.0
      Error/Variance: 0.0
      Observables: [pI]
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.41106719367588934
      Error/Variance: 0.00041058486278825536
      Observables: [pZ]


## 3. Nest bindings

A `CircuitBinding` can contain other bindings. This lets you define an inner
parameter sweep, then supply measurements at the outer level.

### Supply a missing measurement

```python
inner = CircuitBinding([c1, c2], values=[v1, v2])
outer = CircuitBinding(
    [inner, c3],
    measurements=[m1, m2],
    mode=BindingMode.ZIP,
)
```

ZIP pairs `inner` with `m1` and `c3` with `m2`. The inner sweep expands to four
executions, all using `m1`:

| Circuit | Parameter values | Measurement |
| --- | --- | --- |
| `c1` | `v1` | `m1` |
| `c1` | `v2` | `m1` |
| `c2` | `v1` | `m1` |
| `c2` | `v2` | `m1` |
| `c3` | None | `m2` |

### Combine options from both levels

When parameters or measurements are supplied at both levels:

- **Disjoint parameter names** are merged into one mapping.
- **Overlapping parameter names** contribute alternative parameter sets.
- **Measurements at both levels** contribute alternative measurements.

The runnable example below uses `v1` inside and `v2` outside. These mappings
contain the same parameter names, so they remain alternatives. Combined with
`m1` inside and `m1_bis` outside, they produce:

| Circuit | Parameter values | Measurement |
| --- | --- | --- |
| `c1` | `v2` | `m1_bis` |
| `c1` | `v2` | `m1` |
| `c1` | `v1` | `m1_bis` |
| `c1` | `v1` | `m1` |
| `c2` | `v2` | `m1_bis` |

Use `binding.unroll()` to inspect the resolved execution sequence without
submitting any jobs.

In [14]:
binding_embedded = CircuitBinding(
    [CircuitBinding([c1], values=[v1], measurements=m1), c2],
    values=[v2],
    measurements=[m1_bis],
    mode=BindingMode.PRODUCT,
)

print("\n".join(str(c) for c in binding_embedded.unroll()) + "\n")
print(run(binding_embedded, AWSDevice.BRAKET_LOCAL_SIMULATOR))

(QCircuit([U(θ, phi, psi, 0)], label="c1"), {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}, ExpectationMeasure(Observable(pZ, 'Exp1bis_0'), shots=2024, label='Exp1bis'))
(QCircuit([U(θ, phi, psi, 0)], label="c1"), {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}, ExpectationMeasure(Observable(pI, 'Exp1_0'), shots=2024, label='Exp1'))
(QCircuit([U(θ, phi, psi, 0)], label="c1"), {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}, ExpectationMeasure(Observable(pZ, 'Exp1bis_0'), shots=2024, label='Exp1bis'))
(QCircuit([U(θ, phi, psi, 0)], label="c1"), {'θ': 1.0, 'phi': 1.0, 'psi': 1.0}, ExpectationMeasure(Observable(pI, 'Exp1_0'), shots=2024, label='Exp1'))
(QCircuit([H(0)], label="c2"), {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}, ExpectationMeasure(Observable(pZ, 'Exp1bis_0'), shots=2024, label='Exp1bis'))

BatchResult: 5 results
    Result: c1, AWSDevice, BRAKET_LOCAL_SIMULATOR
      Variables: {'θ': 2.0, 'phi': 2.0, 'psi': 2.0}
      Expectation value: -0.42193675889328064
      Error/Variance: 0.00040611134955268456
      Observables: